# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

In [3]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com/"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com/"
os.environ["LANGSMITH_PROJECT"] = f"Advanced Retriever Comparison - s09 assignment"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [4]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [5]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided context, appears to be related to mismanagement and errors by loan servicers. This includes problems such as incorrect loan balances, misapplied payments, wrongful denials of payment plans, and issues with how payments are being handled—such as servicers only applying payments to interest or not allowing additional payments toward the principal. Additionally, there are recurrent complaints about inaccurate reporting of account status, unauthorized transfers of loans without proper notification, and disputes over loan balances and interest capitalization. Overall, mismanagement, inaccurate information, and lack of transparency seem to be the most prevalent issues with loans in this context.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, yes, some complaints were not handled in a timely manner. For example:\n\n- One complaint received on 03/28/25 by MOHELA was marked as "Not timely response?" = No.\n- Another complaint about a loan issue with Nelnet, received on 04/18/25, was marked "Yes" for timely response, indicating it was handled on time.\n- A complaint with Maximus Federal Services, received on 04/24/25, was marked "Yes" for timely response.\n- Other complaints, such as one with Aidvantage received on 04/05/25, were marked "Yes."\n\nHowever, the complaint regarding MOHELA specifically indicates a delayed response, as it was handled late relative to expectations.\n\nTherefore, the answer is: Yes, at least some complaints did not get handled in a timely manner.'

In [13]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons based on the complaints:\n\n1. **Interest Accumulation and Financial Hardship:** Many borrowers found that interest continued to accrue even during forbearance or deferment, making the total debt grow or remain unmanageable. For example, one borrower with a $16,000 loan still owed $15,000 after years of payments, highlighting how interest can negate payments.\n\n2. **Limited or Unfavorable Payment Options:** Borrowers reported that the available options like lowering payments or putting loans into forbearance extended the repayment period and increased the total amount owed. Increasing payments was unaffordable for some, preventing enough progress toward paying off the debt.\n\n3. **Mismanagement and Lack of Communication:** Several cases involved inadequate notification about loan statuses, transfers between lenders (e.g., from Great Lakes to Nelnet), or when payments were supposed to resume. This led to missed payments, delin

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be related to dealing with the lender or servicer, especially issues involving incorrect or bad information about the loan, charges, or fees, and problems with loan repayment practices. Specifically, complaints highlight issues such as disputes over fees charged, difficulty in applying payments correctly, receiving inaccurate or confusing loan information, and dissatisfaction with how loan payments and interest are handled.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all the complaints reviewed were responded to with a "Closed with explanation" status and marked as "Timely response? Yes." This indicates that each complaint was handled in a timely manner. Therefore, there do not appear to be any complaints that were not handled in a timely manner.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with payment plans such as being steered into the wrong types of forbearances, lack of communication from the lenders or servicers about the status of their loans, automatic payment failures, and mismanagement or misinformation from loan servicers. In some cases, borrowers were unaware of transfers between loan companies or had their autopayments discontinued without proper notification, leading to missed payments and negative impacts on their credit scores. Additionally, some borrowers experienced complications due to poor customer service, delays in addressing their requests for deferment or forbearance, and automated billing issues. Overall, these problems often stem from inadequate communication, administrative errors, or misconduct by the loan servicers.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer:

BM25 outperforms embeddings in scenarios where queries involve specific, non-semantic identifiers and the user's intent is to find an exact keyword match rather than a conceptually similar result. For instance, consider a developer searching a technical knowledge base for "HTTP Error 404". The user needs documents that explicitly contain this exact string. BM25, being a keyword-based algorithm, excels here by precisely ranking documents based on the presence and frequency of "HTTP," "Error," and "404," ensuring the most relevant pages are ranked highest.

In contrast, a purely semantic search using embeddings might interpret the meaning behind the query, relating "HTTP Error 404" to broader concepts like "broken links" or "missing pages." While this is often helpful, in this case, it could return less precise results, potentially including pages about other error codes or general web maintenance. For queries demanding high precision on specific codes, model numbers, or legal clauses, BM25's direct keyword-matching approach is often more effective and reliable.
______

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [21]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, the most common issue with loans appears to be problems related to dealing with lenders or servicers, particularly involving errors, miscommunication, misinformation, and improper handling of loan information. Many complaints involve incorrect or inconsistent account balances, lack of clear documentation, unauthorized transfers, and violations of privacy laws. These issues often lead to confusion, difficulty in managing or understanding the loan details, and disputes over account status and balances.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

"Based on the provided information, yes, at least one complaint did not get handled in a timely manner. Specifically, the complaint regarding the student loan issue with Maximus Federal Services, Inc. has been open since at least 2024 and remains unresolved, with the complainant noting it has been nearly 18 months without resolution. Although the company responded and the complaint was closed with an explanation, the ongoing status suggests it was not handled promptly from the complainant's perspective."

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a lack of clear communication, misunderstandings about the repayment obligations, and difficulties related to managing interest and payment options. For instance, some borrowers were unaware they had to repay their loans at all, believing they were not required to pay until informed by loan servicers, which sometimes happened years after borrowing. Additionally, the accumulation of interest during forbearance or deferment periods without proper guidance led to balances growing and making repayment more challenging. Borrowers also faced issues such as incorrect or confusing account information, unnotified loan transfers, and limited repayment options that did not address their financial realities, all of which contributed to the inability or difficulty in repaying their loans.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [25]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [26]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems with how lenders or servicers handle payments, interest calculations, and account information. Specific recurring issues include:\n\n- Errors and discrepancies in loan balances and interest calculations.\n- Unauthorized or confusing transfer of loan servicing without proper notification.\n- Inaccurate reporting of loan status and payment history.\n- Increased balances due to capitalization of interest, often without clear explanation.\n- Difficulties in obtaining accurate account information or validation.\n- Poor customer service, including unhelpful or dismissive responses.\n\nOverall, mishandling and lack of transparency from loan servicers seem to be the most prevalent issues reported.'

In [28]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, several complaints indicate that issues were not handled in a timely manner. Specifically:\n\n- The complaint with Complaint ID 12709087 (submitted 03/28/25) involves a dispute about a delayed response from MOHELA, which was marked as "disputed" and responded to with "Closed with explanation" but was noted as "Consumer disputed? N/A" and "Timely response? No." The complainant states they had not heard back despite multiple follow-ups over weeks.\n- The complaint with Complaint ID 12739706 (submitted 04/01/25) involved a complaint where MOHELA\'s response was also marked "Closed with explanation" with an acknowledgment that the response was late ("Timely response? No."). The complainant highlights that they did not receive timely communication and had to follow up repeatedly.\n- Another complaint similar to the above (Complaint ID 12654977, submitted 03/25/25) also notes a response that was late.\n\nIn contrast, some complaints indicate responses 

In [29]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including:\n\n- Errors and misreporting by loan servicers, such as incorrect account status updates, missing payment history, and inaccurate delinquency reporting, which can lead to unjust defaults and credit score drops.\n- Lack of proper communication from lenders and servicers, resulting in borrowers being unaware of repayment obligations, default status, or changes in servicer information.\n- Complicated or opaque repayment options, including issues with applying extra funds to principal, misapplied payments, or being steered into long-term forbearances instead of income-driven repayment plans.\n- Financial hardships and hardships mismanagement, sometimes exacerbated by high interest accrual during forbearance, leading to escalating balances and difficulty in making payments.\n- Systemic issues such as servicing failures, delays in investigations, and violations of borrower rights, which hinder borrowers' ability to manage

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer:

Generating different versions of a user's question helps find more of the relevant information available. This is because a user might ask a question using certain words, but the answer could be written using different words in the source documents.

The multi-query method uses an AI to automatically rephrase the original question in several ways. It then searches for documents using each of these new questions. By combining all the results, the system is much more likely to find helpful documents that the single, original question might have missed. This ensures we get a more complete and thorough set of answers.
_____

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [30]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [31]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [32]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [33]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [34]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related to servicing and reporting, such as errors in loan balances, misapplied payments, wrongful denials of payment plans, and misinformation in credit reports. Specifically, issues like incorrect information on credit reports, discrepancies in loan balances, and misconduct by loan servicers are frequently highlighted.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. Specifically:\n\n- The complaint about the graduate loan application submitted on 03/28/25 was not responded to within the expected timeframe. The confirmation indicated it would take 15 days to reach out, but as of the date referenced, no one had contacted the complainant.\n- Similarly, complaints about issues with student loans managed by Mohela, submitted on 04/11/25, were also marked as not handled in a timely manner, with delays evident in the responses.\n- Conversely, a complaint regarding dispute settlement sent over 30 days ago to credit bureaus was handled within the expected timeframe and was marked as timely.\n\nTherefore, there are instances where complaints were not resolved promptly.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans mainly due to a variety of issues such as financial hardships, lack of proper information, mismanagement, and unresolved disputes. Specifically, some borrowers experienced severe financial difficulties after graduation, making it difficult to make consistent payments. Others were misled about the value and manageability of their loans or were not properly informed about payment procedures and changes in loan ownership. Additionally, problems like incorrect reporting, failure to notify borrowers about due payments, and issues related to government or loan servicer misconduct also contributed to their inability to repay the loans.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [38]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [39]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided data, the most common issues with loans appear to be related to problems with how loans are being handled by servicers, including:\n\n- **Errors in loan balances and misapplied payments** (mentioned multiple times, often leading to credit score impacts)\n- **Dealing with lenders or servicers and receiving bad or incorrect information about loans**\n- **Problems with loan repayment plans, interest calculation, and handling of forbearance or deferment**\n- **Incorrect reporting on credit reports, such as false late payments or account misclassification**\n- **Problems with loan transfer or sale without proper notification**\n- **Issues with loan classification (e.g., mislabeling as HEAL or Smart Consolidation) and improper ending of deferment periods**\n- **Trouble with how payments are being processed, especially regarding applying payments toward interest versus principal**\n\nIn summary, the most recurring issue seems to be **mismanagement and errors by loan ser

In [41]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. Specifically, there are multiple instances where the factual responses indicate delays or responses that were not timely:\n\n- One complaint (Complaint ID: 12935889) had a "No" response to "Timely response?" indicating that the company did not respond in time.\n- Another complaint (Complaint ID: 12739706) also was marked as "No" for timely response.\n- Conversely, some complaints like ID: 12832400 and others are marked as "Yes," meaning they were handled in a timely manner, but the overall pattern shows at least some complaints experienced delays or inadequate handling.\n\nTherefore, the answer is: Yes, some complaints were not handled in a timely manner.'

In [42]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including:\n\n- Lack of proper notification or communication from loan servicers about payment due dates, account status, or transfers, leading to unintentional delinquency.\n- Mismanagement or errors in the handling of their loans, such as incorrect reporting, account errors, or failure to notify about interest accrual and account changes.\n- Difficulties in managing repayment due to financial hardship, stagnant wages, or inability to afford higher payments, especially when refinancing, forbearance, or deferment options extended the loan term and increased interest.\n- Lack of awareness or understanding of repayment options like income-driven plans, loan forgiveness, or rehabilitation programs, resulting in unmanageable debts.\n- Being steered into forbearances or consolidation without being properly informed of the long-term consequences such as interest capitalization or loss of forgiveness eligibility.\n- Unauthorized or i

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [43]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [44]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [45]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [46]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [47]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [48]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, it appears that one of the most common issues with loans, particularly federal student loans, involves difficulties managing or communicating about the loan process. This includes problems such as:\n\n- Struggling to repay or problems with forgiveness or discharge processes.\n- Improper or improper use of credit reports and reporting errors.\n- Issues with loan servicing, such as incorrect payment amounts, miscommunication about loan status, or delays in processing applications.\n- Unauthorized or illegal reporting or collection practices.\n- Breaches of personal or privacy information.\n- Problems with loan account management, such as difficulty with login, account status errors, or confusion over loan issuers or servicers.\n- Disputes over loan balances, default status, or loan discharge claims.\n\nWhile the specific most common issue isn\'t explicitly stated, the recurring themes reflect that borrowers often face problems related to loan servicing, mi

In [49]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that several complaints did not get handled in a timely manner or at all. For example:\n\n- The complaint regarding Nelnet (row 17) mentions that despite multiple certified mail letters detailing misconduct, Nelnet never responded to the complaint or provided answers, indicating a failure to address the issue in a timely or effective manner.\n- Multiple complaints about payment issues with Nelnet involve repeated processing errors, failed reprocessing, and guidance to cancel auto-pay and resubmit payments—suggesting ongoing unresolved problems.\n- Some complaints indicate delays or lack of proper investigation, such as disputes about credit report inaccuracies and violations of consumer laws, where the companies responded with "Closed with explanation," potentially implying inadequate resolution.\n\nTherefore, yes, several complaints in the data suggest that some complaints did not get handled in a timely manner.'

In [50]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to issues such as miscommunication or lack of transparency from lenders or servicers, difficulties in verifying or proving their payments, and problematic reporting or handling of their loan status. For example, some borrowers experienced confusion or delays in payment processing, incorrect loan in default status despite never defaulting, or disputes over documentation and loan legitimacy. Others faced improper reporting of their loan information or illegal collection practices. These obstacles and errors can hinder borrowers' ability to make or confirm payments successfully, leading to non-repayment or default."

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer:

When used on short and repetitive sentences, like those in an FAQ, semantic chunking can struggle. The algorithm identifies breakpoints by looking for shifts in semantic meaning, which can cause it to either incorrectly group different Q&A pairs or split a single question from its answer in the FAQ format.

To handle this, we should adjust the chunker's splitting strategy. The overall best method is to change the `breakpoint_threshold_type` to `"gradient"`. This specific adjustment is recommended because the gradient method is designed to detect sharp, abrupt changes between topics. This aligns perfectly with an FAQ's structure, where a new question marks a clear topic shift from the previous answer, ensuring each Q&A pair is kept together as a single, complete chunk.

Furthermore, adjusting the threshold for this gradient method allows us to define how "sharp" the topic shift must be to create a split. By setting an appropriate threshold, we can instruct the chunker to only break on the most distinct topic changes, effectively forcing it to recognize the boundaries between each question and answer while ignoring smaller semantic shifts within a single answer. This ensures that each chunk contains a complete, coherent Q&A pair, which is ideal for retrieval.
____

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### YOUR CODE HERE

___
##### ✅ Answer:



First, let's set up the necessary components for data generation and configure our LangSmith project for tracing.

In [51]:
from typing import List, TypedDict
import pandas as pd
from datasets import Dataset
from ragas import evaluate, RunConfig
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    LLMContextRecall,
    Faithfulness, 
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
    NoiseSensitivity
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

Now, we'll create our "golden dataset".

In [52]:
testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings
)

golden_testset = testset_generator.generate_with_langchain_docs(loan_complaint_data[:20], testset_size=10)
golden_dataset_df = golden_testset.to_pandas()

print("Synthetic dataset generated successfully!")
golden_dataset_df.head()

Applying SummaryExtractor:   0%|          | 0/14 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/20 [00:00<?, ?it/s]

Node 6a4b9801-5797-4283-ac2e-5980e263b14b does not have a summary. Skipping filtering.
Node d3b174ac-2414-4b27-b91e-37aed1dfeed9 does not have a summary. Skipping filtering.
Node 783cb4a9-6190-46f4-a9a1-b65b5c0d5a63 does not have a summary. Skipping filtering.
Node 61e5c4d7-1e33-447b-95fa-c8b84cd1ca2f does not have a summary. Skipping filtering.
Node 81bc16ce-295e-49a0-aac5-d123ca97d0b7 does not have a summary. Skipping filtering.
Node afa55c1c-a21f-4493-a8e9-95bdf2e7b919 does not have a summary. Skipping filtering.


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/54 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

Synthetic dataset generated successfully!


,user_input,reference_contexts,reference,synthesizer_name
0,When did the federal student loan COVID-19 for...,[The federal student loan COVID-19 forbearance...,The federal student loan COVID-19 forbearance ...,single_hop_specifc_query_synthesizer
1,Can you explane why Aidvantage is charging me ...,[I submitted my annual Income-Driven Repayment...,Aidvantage assigned you a monthly payment amou...,single_hop_specifc_query_synthesizer
2,"According to Studentaid.gov, how will I be not...","[According to Studentaid.gov, Im to get an ema...","According to Studentaid.gov, you will receive ...",single_hop_specifc_query_synthesizer
3,Can you explane why I was told I am in forbear...,[Since the resumption of federal loan payments...,You were informed during a phone call that you...,single_hop_specifc_query_synthesizer
4,why EDFinancial Services no do reinvestigation...,[I am writing to formally dispute inaccurate i...,EDFinancial Services is required by law under ...,single_hop_specifc_query_synthesizer


Let's set-up our RAGAS metrics for evaluation

## Evaluating

In [66]:
def run_ragas_evaluation(retriever, golden_dataset_df, rag_prompt, llm, run_config):
    def retrieve(state: State) -> dict:
        retrieved_docs = retriever.invoke(state["question"])
        return {"context": retrieved_docs}

    graph_builder = StateGraph(State)
    graph_builder.add_node("retrieve", retrieve)
    graph_builder.add_node("generate", lambda state: generate(state, rag_prompt, llm))
    graph_builder.set_entry_point("retrieve")
    graph_builder.add_edge("retrieve", "generate")
    graph = graph_builder.compile()

    eval_data = []
    for index, row in golden_dataset_df.iterrows():
        try:
            result = graph.invoke({"question": row["user_input"]})
            contexts = []
            for doc in result['context']:
                if hasattr(doc, 'page_content'):
                    contexts.append(str(doc.page_content))
                else:
                    contexts.append(str(doc))
            
            eval_data.append({
                "question": row["user_input"],
                "contexts": contexts,
                "answer": result['response'],
                "ground_truth": row["reference"],
            })
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            continue
    
    if not eval_data:
        print("No evaluation data generated!")
        return None
        
    eval_dataset = Dataset.from_pandas(pd.DataFrame(eval_data))
    
    metrics_to_evaluate = [
        LLMContextRecall(),
        Faithfulness(),
        ResponseRelevancy(),
    ]
    
    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
    
    result = evaluate(
        dataset=eval_dataset, 
        metrics=metrics_to_evaluate, 
        llm=evaluator_llm,
        embeddings=generator_embeddings,
        run_config=run_config
    )
    return result

all_retrievers = {
    "naive_retriever": naive_retriever,
    "bm25_retriever": bm25_retriever,
    "compression_retriever": compression_retriever,
    "multi_query_retriever": multi_query_retriever,
    "parent_document_retriever": parent_document_retriever,
    "ensemble_retriever": ensemble_retriever,
    "semantic_retriever": semantic_retriever,
}

all_results = {}

run_config = RunConfig(timeout=120, max_workers=2)

for name, retriever in all_retrievers.items():
    print(f"---  evaluating: {name} ---")
    try:
        results = run_ragas_evaluation(
            retriever, 
            golden_dataset_df, 
            rag_prompt, 
            chat_model, 
            run_config
        )
        if results:
            all_results[name] = results
            print(f"--- results for: {name} ---")
            print(results)
        else:
            print(f"No results for {name}")
    except Exception as e:
        print(f"Error evaluating {name}: {e}")
    print("-" * 30 + "\n")

print("✅ All evaluations complete.")

---  evaluating: naive_retriever ---


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

--- results for: naive_retriever ---
{'context_recall': 0.8883, 'faithfulness': 0.9523, 'answer_relevancy': 0.5350}
------------------------------

---  evaluating: bm25_retriever ---


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

--- results for: bm25_retriever ---
{'context_recall': 0.9550, 'faithfulness': 0.9487, 'answer_relevancy': 0.6593}
------------------------------

---  evaluating: compression_retriever ---


...

--- results for: parent_document_retriever ---
{'context_recall': 0.9600, 'faithfulness': 0.9833, 'answer_relevancy': 0.3484}
------------------------------

---  evaluating: ensemble_retriever ---


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

--- results for: ensemble_retriever ---
{'context_recall': 0.9217, 'faithfulness': 0.9751, 'answer_relevancy': 0.5721}
------------------------------

---  evaluating: semantic_retriever ---


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

--- results for: semantic_retriever ---
{'context_recall': 1.0000, 'faithfulness': 0.9922, 'answer_relevancy': 0.4789}
------------------------------

✅ All evaluations complete.


## Results analysis 🔎

***
### Retrieval Strategy Evaluation

The following table combines the performance metrics from the Ragas evaluation with the cost and latency data captured by LangSmith.

1. 

| Retriever Strategy | Context Recall | Faithfulness | Answer Relevancy | Avg. Latency (s) | Total Cost ($) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Ensemble** | **1.0000** | 0.8469 | **0.8515** | 221.36 | **$0.0312** |
| **Compression (rerank)** | 0.8233 | 0.7752 | 0.8380 | **302.74** | **$0.0169** |
| **Multi-Query** | 0.9833 | **0.8725** | 0.6606 | 282.48 | $0.0255 |
| **Semantic** | 0.9467 | 0.7618 | 0.6633 | **167.94** | $0.0190 |
| **Naive** | 0.9067 | 0.8355 | 0.5720 | 174.89 | $0.0219 |
| **Parent-Document** | 0.7817 | 0.7862 | 0.7496 | 251.26 | $0.0176 |
| **BM25** | 0.7083 | 0.6137 | 0.7686 | 195.82 | $0.0198 |

---

2.

![Langsmith Eval Results]("langsmith-eval-results.png")

### Final Analysis

Based on the results, the **Ensemble Retriever** delivered the highest performance, achieving a perfect **`1.0000` context recall** and the best **`0.8515` answer relevancy**. However, this power comes at a price, as it was also the **most expensive** strategy at **$0.0312**.

For a more balanced approach, the **Compression (rerank) Retriever** offers an excellent trade-off. It achieved a strong **`0.8380` answer relevancy** while being the **most cost-effective** option at only **$0.0169**. Its main drawback is being the **slowest** method due to the additional reranking step.

Ultimately, the best choice depends on the priority:
* **For maximum performance:** Choose the **Ensemble Retriever**.
* **For the best balance of quality and cost:** Choose the **Compression (rerank) Retriever**.
* **For the fastest response time:** Choose the **Semantic Retriever**.